# L14a: Updating the Single Index Model and Paper Trading
Previously, we held each stock's market sensitivity at its 2014 to 2024 estimate. How do we keep the estimate current as new trading days arrive, and how do we send orders for the stocks we pick?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Update the single index model every day:__ Update each stock's baseline and market sensitivity with each new trading day by exponentially weighted least squares, starting from a fit on past data. Describe how the decay sets how fast old data fades.
> * __Pick stocks with the updated preference weights:__ Compute the preference weights from the updated estimates, and pick a new γ basket once a month and whenever the market turns. Compare the tilt of the preference weight with the betting-against-beta portfolio.
> * __Place paper trades through a trading API:__ Describe a paper trading account, the time in force of an order, and what happens to an order after we send it. Send orders for the γ basket to a paper account, and follow each order until it fills or we cancel it.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
>
> * Exponentially weighted least squares updates each stock's single index estimates every trading day. Recent days count the most, because the weight of a day of data halves every 63 trading days.
> * The preference weight uses the updated estimates and the market state $\xi_{t}$. When the market falls, it favors stocks with a low market sensitivity, the bet of betting against beta.
> * A trading API lets a program check an account, read prices, and send orders. A paper account trades simulated money at real market prices.
>
> __For understanding__ (you will not need to reproduce or memorize these):
>
> * The derivation of the daily update from the weighted least-squares problem.
> * The trading method's rules for picking the γ basket: the sector quota, the floor on the market sensitivity, and the market-turn rule.
> * How the MCP server turns a request in plain English into calls to Alpaca's API.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L14a) lists this lecture's symbols with their meanings and units.

Let's get started!

___

## Examples

__We will work through these examples in lecture:__

> [▶ Build a synthetic market of many stocks](CHEME-5660-L14a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we use the sector factor model to build a universe of `SPY` and 375 stocks from all 11 sectors. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data, and the stylized facts of one synthetic stock, with and without jumps, with its real growth rates. Then we save the fitted model. We pick up where we left off.

> [▶ Update the single index model and the preference weights](CHEME-5660-L14a-Example-Updating-SIM-Gamma-Fall-2026.ipynb). In this example, we update the single index model of every stock with each trading day of 2025 and 2026, starting from the 2014 to 2024 fit. We compute the preference weights from the updated estimates and pick the γ basket once a month and whenever the market turns. Then we compare it with the basket that the fixed 2014 to 2024 estimates pick.

> [▶ Paper trade the γ basket](alpaca-mcp/README.md). In this example, we connect Claude Code to an Alpaca paper account through a Model Context Protocol (MCP) server. We check the account and the quotes of the γ basket, place limit orders for its stocks, and follow each order until it fills or we cancel it.

___

## Review: Synthetic Markets with Jump Hidden Markov Models
Previously, we built synthetic markets from jump hidden Markov models. The model bins the daily growth rate into states of equal probability and estimates the transition matrix by counting moves between states.

Each simulated day, the chain takes one step. The model then draws the day's growth rate from the Student-t distribution of the new state:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-JumpHMM-Schematic.svg" width="880" style="max-width:100%; height:auto;" alt="A row of hidden states numbered 1, 2, through k, through N minus 1 and N, ordered from the most negative to the most positive daily growth rate, with double-headed arrows between neighbors. States 1 and 2 are red and braced as the most negative states S minus, and states N minus 1 and N are blue and braced as the most positive states S plus. A bell curve above each state is its emission, and state k emits mu k plus sigma k times a Student-t variable. Under state k, a note reads ordinary day: step with the estimated transition matrix. A dashed arrow from state k, marked no episode under way, leads up to a box: jump episode, starts with probability epsilon, lasts a Poisson number of days with mean lambda. From the box, a red arrow runs to the most negative states, taken on each episode day with probability p neg, and a blue arrow runs to the most positive states, taken with probability 1 minus p neg."/>
    </center>
</div>

Jump episodes start at random and last several days. On each episode day, the state comes from the most negative or the most positive states. These runs of large moves add the volatility clustering that the chain alone misses. Then we extended the single-asset jump hidden Markov model to many stocks that move together. We started from the single index model and added a sector factor:

> __Definition (sector factor model):__ A sector factor model adds one factor for each sector of the [Global Industry Classification Standard (GICS)](https://www.msci.com/our-solutions/indexes/gics). 
> 
> Each sector factor describes how the single index residuals of the stocks in its sector move together. Let's model the daily growth rate of stock $i$ in sector $s$ as:
>
> $$
> g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \overbrace{\underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector}} + \underbrace{u_{i}(t)}_{\text{stock-specific}}}^{\text{single index residual }\varepsilon_{i}(t)}
> $$
>
> Here $g_{m}(t)$ is the growth rate of `SPY`, $f_{s}(t)$ is the growth rate of sector $s$ (with the market part removed), and $u_{i}(t)$ is the growth rate that neither the market nor the sector explains.

The market sensitivity $\beta_{i}$ and the sector loading $\delta_{i}$ set how much stock $i$ moves with the market and with its sector. We built the many-stock model from single-asset jump hidden Markov models: one for the market, one for each sector, and one for each stock.

Let's pick up where we left off, and build a synthetic market of many stocks:

> __Example:__
>
> [▶ Build a synthetic market of many stocks](CHEME-5660-L14a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we use the sector factor model to build a universe of `SPY` and 375 stocks from all 11 sectors. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data, and the stylized facts of one synthetic stock, with and without jumps, with its real growth rates. Then we save the fitted model.

__Today:__ We update the single index model with each new trading day, pick stocks with the updated preference weights, and send orders for those stocks to a paper account.

___


## Company Profile: Alpaca
[Alpaca](https://alpaca.markets/about) is a brokerage built for programs that trade. Its customers send orders through an application programming interface (API), a set of requests that a program sends to Alpaca's servers. Yoshi Yokokawa and Hitoshi Harada founded the company in 2015, and it became a registered broker-dealer in 2018.

> __What does the firm do?__
>
> * __Trading for programs:__ Alpaca's [Trading API](https://docs.alpaca.markets/docs/trading-api) lets a program check an account, read prices, and send orders. Every user gets a free [paper trading account](https://docs.alpaca.markets/docs/paper-trading), which trades simulated money at real market prices.
> * __Brokerage for other apps:__ Alpaca's Broker API lets other firms build trading apps on top of Alpaca's accounts. In [January 2026](https://alpaca.markets/blog/alpaca-raises-150-million-at-a-1-15b-valuation-to-build-the-global-standard-for-brokerage-infrastructure/), Alpaca raised 150 million USD at a valuation of 1.15 billion USD, and it reported over 9 million brokerage accounts in more than 40 countries.
> * __AI assistants that trade:__ Alpaca maintains an [official MCP server](https://github.com/alpacahq/alpaca-mcp-server), which lets an AI assistant such as Claude check an account and place orders. In [July 2026](https://alpaca.markets/blog/alpaca-raises-135-million-to-scale-agent-first-brokerage-infrastructure-for-tokenized-markets-and-ai-native-financial-services), Alpaca raised 135 million USD to build what it calls "agent-first" brokerage infrastructure. It reported that its monthly active API users grew nearly four times over the previous six months.

__Explore further:__

* __Jobs:__ Alpaca's [hiring page](https://alpaca.markets/hiring) says the company works fully remote, with team members in more than 20 countries. In November 2026, its [job board](https://job-boards.greenhouse.io/alpaca) listed roles in engineering, data science, compliance, and brokerage operations.
* __Video:__ Watch [Backend Operations for a Trade](https://www.youtube.com/embed/KiWT1qq59Gc) on the firm's YouTube channel. In this 2023 video, Alpaca's VP of Operations walks through what happens after a customer places a trade.

__Connection to today's lecture:__ We'll send orders for the γ basket to an Alpaca paper account through our own MCP server. The paper account fills our orders at real market prices with simulated money, which lets us test the trading method before it risks real money.

___

## Updating the Single Index Model
Previously, we fit the single index model of each stock once, on 2014 to 2024, and held its estimates fixed. However, a stock's market sensitivity changes over time. By late 2026, most stocks in our universe moved less with the market than their 2014 to 2024 estimates say. We keep the estimates current by updating them with each new trading day, giving recent days the most weight:

> __Exponentially weighted least squares (EWLS):__ On trading day $t$, the updated baseline $\alpha_{i}(t)$ and market sensitivity $\beta_{i}(t)$ of stock $i$ solve the weighted least-squares problem:
>
> $$
> \left(\alpha_{i}(t),\beta_{i}(t)\right) = \arg\min_{a,\,b}\Biggl[\underbrace{\sum_{k=t_{0}+1}^{t}\phi^{\,t-k}\left(g_{i}(k)-a-b\,g_{m}(k)\right)^{2}}_{\text{weighted data}} + \underbrace{\phi^{\,t-t_{0}}N_{0}\,\mathbb{E}_{0}\left[\left(g_{i}-a-b\,g_{m}\right)^{2}\right]}_{\text{prior}}\Biggr]
> $$
>
> where $g_{i}(k)$ and $g_{m}(k)$ are the daily growth rates of stock $i$ and `SPY` on day $k$ (1/yr), $a$ and $b$ are a candidate baseline and market sensitivity, and $t_{0}$ is the last day of the fitting period. The prior weight $N_{0}=63$ counts the 2014 to 2024 fit as 63 days of data, and $\mathbb{E}_{0}$ is the expected value under that fit. The decay $\phi=2^{-1/63}$ halves the weight of a day of data every 63 trading days, about one quarter.
>
> Each part of the problem has a job:
>
> * __Weighted data:__ Each day, the weight of every earlier day shrinks by the factor $\phi$, and the new day enters with weight one. Because recent days count the most, the estimates follow changes in how the stock moves with the market.
> * __Prior:__ The prior term is the expected squared residual under the 2014 to 2024 fit. On the first day, the prior alone gives the 2014 to 2024 estimates. The weight of the prior decays at the same rate as the weight of the data, and after a year of trading the prior carries little weight.
> * __No look-ahead:__ The estimates on day $t$ use the growth rates through day $t$ and nothing later. The trading method decides after the close of day $t$ and trades at the next open.

Solving this problem from scratch each day would mean storing every past day, so we use [the EWLS recursion](advanced/ewls/CHEME-5660-L14a-Derivation-EWLS-Update-Fall-2026.ipynb), which updates the estimates from a few running sums.

### Preference Weights with Updated Estimates
Previously, we computed the preference weight $\gamma_{i}(t)$ of each stock from its single index estimates and two market inputs from `SPY`: the recent market growth rate $\tilde{g}_{M,t}$ (1/yr) and the market state $\xi_{t}$. The trading method of [Varner (2026)](https://arxiv.org/abs/2611.03022) uses the same preference weight to choose its stocks, with the updated estimates in place of the fixed ones:

> __Preference model (updated estimates):__ For stock $i$ on day $t$, the preference weight is given by:
>
> $$
> \boxed{\gamma_{i}(t) = \tanh\left(h\,\frac{\alpha_{i}(t)+\beta_{i}(t)\,\tilde{g}_{M,t}}{\max\left(|\beta_{i}(t)|,\,\beta_{\text{floor}}\right)^{\xi_{t}}}\right)}
> $$
>
> where the horizon $h=1$ year makes the argument of the $\tanh$ dimensionless, and the floor is $\beta_{\text{floor}}=0.1$. The market state $\xi_{t}$ is positive when the market has recently been falling, and negative when it has been rising. With a fixed $\beta_{i}\geq\beta_{\text{floor}}$, this is the preference weight we used before.
>
> The trading method makes three changes to how we use the preference weight:
>
> * __Updated estimates:__ The estimates $\alpha_{i}(t)$ and $\beta_{i}(t)$ come from the daily EWLS update. The preference weight follows the changes in each stock's market sensitivity.
> * __A floor on the market sensitivity:__ Some stocks have an updated market sensitivity near zero, or below zero. The denominator uses $|\beta_{i}(t)|$ with a floor, which keeps the preference weight defined for every stock. The numerator keeps the sign of $\beta_{i}(t)$, which lowers the preference weight of a stock that moves against a growing market.
> * __Picks at market turns:__ The method keeps the three stocks with the highest preference weights in each sector, which we call the __γ basket__. It picks a new basket every 21 trading days, and whenever the market turns. The market turns on the day $\xi_{t}$ moves to the other side of zero from its value on the last pick day.

### The Market Hedge
The trading method can also run with a __market hedge__. The hedge limits the market sensitivity of the whole portfolio, and the limit drops as the market falls:

> __Market hedge:__ On each day $t$ that the trading method sets its holdings, the hedge requires:
>
> $$
> \boxed{\beta_{P}(t) = \sum_{i}w_{i}(t)\,\beta_{i}(t) \leq \max\left(1-\xi_{t},\,-0.5\right)}
> $$
>
> where $\beta_{P}(t)$ is the market sensitivity of the portfolio, and $w_{i}(t)$ is the fraction of the portfolio's value in holding $i$, negative for a short. Cash has a market sensitivity of zero.
>
> In a flat market ($\xi_{t}=0$), the limit is one, the market sensitivity of `SPY`. The limit reaches zero at $\xi_{t}=1$. Because we scaled $\xi_{t}$ so that $|\xi_{t}|\leq 1$ on 99% of training days, only a steep fall takes the limit below zero.
>
> To meet the limit, the method can shift value toward the basket's stocks with a low market sensitivity, hold cash, or buy up to five stocks whose market sensitivity is zero or negative.
>
> When the limit is below zero, the method also sells `SPY` short to cover the part below zero. Because `SPY` has a market sensitivity of one, a limit of $-0.2$ gives a short worth 20% of the portfolio's value.

__How does the trading method compare with betting against beta?__ Previously, we met [AQR Capital Management](https://www.aqr.com/Our-Firm/About-Us), where Andrea Frazzini and Lasse H. Pedersen proposed the [betting-against-beta portfolio](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta) in 2014. Their portfolio always buys stocks with a low market sensitivity and sells stocks with a high one, in amounts that make its market sensitivity zero. The trading method makes the same bet only when the market falls:

* __Signal__: When the market falls ($\xi_{t}>0$), the denominator is above one for a stock with $|\beta_{i}(t)|>1$ and below one for a stock with $|\beta_{i}(t)|<1$. The preference weight then favors stocks that move less with the market. When the market rises ($\xi_{t}<0$), the tilt reverses, and the preference weight favors stocks that move more with the market.
* __Long-only basket__: The γ basket keeps the three stocks with the highest preference weights in each sector. A stock with a low preference weight is left out of the basket, not sold short.
* __Market hedge__: With the hedge on, the limit on $\beta_{P}(t)$ drops to zero, the market sensitivity of betting against beta, at $\xi_{t}=1$. After a steeper fall, the trading method sells `SPY` short, whereas betting against beta sells stocks with a high market sensitivity.

Let's update the estimates of every stock in our universe and pick the γ basket:

> __Example:__
>
> [▶ Update the single index model and the preference weights](CHEME-5660-L14a-Example-Updating-SIM-Gamma-Fall-2026.ipynb). In this example, we update the single index model of every stock with each trading day of 2025 and 2026, starting from the 2014 to 2024 fit. We pick the γ basket once a month and whenever the market turns, and compare it with the basket that the fixed estimates pick.

The example ends with the γ basket of the last pick day in our data. Next, we send orders for the stocks of a basket to a paper trading account.

___

## Paper Trading with Alpaca
In a trading app, we send an order by tapping a button. A trading method decides its trades in a program, and the program sends its orders through Alpaca's API. We'll send our orders to a paper account, which lets us test the trading method before it risks real money:

> __Definition (paper account):__ A paper account trades simulated money at real market prices. Alpaca fills each paper order against the latest quotes, and the order never reaches an exchange. A live account sends the same orders to the market with real money.

A paper fill can differ from a live fill of the same order. The paper account does not simulate how our order moves the price, the order's place in line behind other orders, regulatory fees, or dividends. Its free quotes also come from one exchange, IEX, and can show a wider bid-ask spread than the national best bid and offer.

We'll ask an AI assistant to make the calls to the API. A __Model Context Protocol (MCP) server__ is a program that gives an AI assistant, such as Claude Code, one tool for each request to Alpaca. The server holds the account's key and secret, which work like a user name and a password, and the assistant never sees them:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L14a-PaperTrading-Schematic.svg" width="1000" style="max-width:100%; height:auto;" alt="A laptop running the AI assistant, Claude Code, shows our request, Buy 1 share of KO with a limit order at the bid, and the assistant asking us to approve the tool call submit_order. An arrow labeled tool call runs to the MCP server, a rack of three units labeled get_latest_quotes, submit_order, and get_order, with an orange key labeled key plus secret. An arrow labeled API request runs to the Alpaca paper account, which holds simulated cash and fills each order against the latest market quotes from IEX. Gray arrows carry the order status and the result back. Below, the life of an order: a sent order is new, then partially_filled, then filled, or goes from new straight to filled. A new or partially filled order is canceled when we cancel it or when a day order reaches the close, and an order that fails a check is rejected."/>
    </center>
</div>

Previously, we defined market and limit orders. A market order fills right away at the best available price, but we do not know that price in advance. A limit order sets the worst price we accept, but it may not fill.

Each order also has a __time in force__, which sets how long an unfilled order stays open. A `day` order is canceled at the close if it has not filled, and a good-'til-canceled (`gtc`) order stays open until it fills or we cancel it. After we send an order, its status tells us where the order stands:

* __Open:__ An order is `new` once Alpaca accepts it. It is `partially_filled` once some of its shares trade, and we can still cancel the rest.
* __Done:__ An order is `filled` when all of its shares trade. It is `canceled` when we cancel it, or when a `day` order reaches the close without filling.
* __Rejected:__ An order is `rejected` when it fails a check before it can trade, for example a buy order that costs more than the account can spend.

A trading method reads the status of each order before its next decision, because an unfilled limit order leaves the method holding fewer shares than it planned. Let's place orders for the γ basket in a paper account:

> __Example:__
>
> [▶ Paper trade the γ basket](alpaca-mcp/README.md). In this example, we connect Claude Code to an Alpaca paper account through an MCP server. We check the account and the quotes of the γ basket, place limit orders for its stocks, and follow each order until it fills or we cancel it.

___

## Optional Advanced Material
> [▶ Derive the daily update of the single index model](advanced/ewls/CHEME-5660-L14a-Derivation-EWLS-Update-Fall-2026.ipynb). In this notebook, we derive the running sums that the exponentially weighted least squares update carries from one day to the next. We solve them for the updated estimates and the residual standard deviation, and show how the 2014 to 2024 fit seeds the sums.

___

## Summary
We updated the single index model of every stock with each trading day, picked the γ basket from the updated preference weights, and sent orders for a basket to a paper account.

> __Key Takeaways:__
>
> * __Updating the single index model:__ Exponentially weighted least squares updates each stock's baseline and market sensitivity with every trading day, starting from the 2014 to 2024 fit. Over 2025 and 2026, the market sensitivity of most stocks fell below their fixed estimates.
> * __Updated preference weights:__ The trading method picks the stocks with the highest updated preference weights in each sector, once a month and whenever the market turns. When the market fell, the γ basket favored stocks with a low market sensitivity, as betting against beta does.
> * __Paper trading:__ A trading API lets a program read prices and send orders. We sent limit orders for the γ basket to a paper account, which trades simulated money at real market prices. Because a limit order may not fill, we followed each order until it filled or we canceled it.

The γ basket says which stocks to consider. Deciding how many shares of each stock to buy, and when to trade, is the rest of the trading method.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___